#### ◆ Library Imports

The required Python libraries are imported to support the development of the ByT5 text normalization model. These libraries provide functionalities for model loading, tokenization, dataset preparation, training configuration, and GPU-based computation.

In [12]:
import time                             # Measure training and inference execution time
import json                             # Save experiment results in JSON format
import sys                              # System modules for handling file paths and project imports
import os                               # Operating system utilities for handling file paths and directories
import torch                            # PyTorch library for deep learning and GPU computation
import numpy as np                      # NumPy library for numerical operations
import pandas as pd                     # Data analysis and summary of dataset statistics
import matplotlib.pyplot as plt         # Plot training and validation loss curves
from transformers import (              # Hugging Face Transformers components for model loading, tokenization, training and data collation
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    TrainingArguments,
    Trainer
)                                       
from datasets import Dataset            # Hugging Face Dataset class for creating datasets compatible with the Trainer API
from utils.metrics import calculate_all_metrics

c:\Users\ae\AppData\Local\miniconda3\envs\dev\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'jiwer'

#### ◆ Project setup

The project root is added to Python's search path to allow the notebook to access shared utility functions stored in the `utils` directory.

In [ ]:
project_root = os.path.abspath("../../")
sys.path.append(project_root)

#### ◆ Environment verification

Check the current project directory and confirm whether GPU acceleration is available through CUDA.

In [ ]:
# Display the current working directory
print("Current folder:")
print(os.getcwd())

# Check whether CUDA-enabled GPU is available
print("CUDA available:")
print(torch.cuda.is_available())

#### ◆ Dataset loading

Training set (`train`), Validation set (`val`), Test set (`test`)

In [ ]:
from utils.dataset_utils import load_dataset_splits

datasets = load_dataset_splits(
    "../../data/dataset_split"
)

train_data = datasets["train"]
val_data = datasets["val"]
test_data = datasets["test"]

#### ◆ Convert to HuggingFace dataset

In [ ]:
train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)
test_dataset = Dataset.from_list(test_data)

#### ◆ Load ByT5-small

In [ ]:
# Define the pretrained ByT5-small model
model_name = "google/byt5-small"

# Load the ByT5 tokenizer for converting text into tokens
tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

# Load the ByT5 sequence-to-sequence model
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name
)

#### ◆ Tokenization function

In [ ]:
MAX_LENGTH = 256

def preprocess_function(examples):

    inputs = examples["input"]
    targets = examples["standardized"]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_LENGTH,
        truncation=True
    )

    labels = tokenizer(
        targets,
        max_length=MAX_LENGTH,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

#### ◆ Apply tokenization

In [ ]:
tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True
)

tokenized_val = val_dataset.map(
    preprocess_function,
    batched=True
)

#### ◆ Remove original text columns

After tokenization, the original dataset columns (`id`, `input`, and `standardized`) are no longer required for model training. Only the tokenized inputs (`input_ids`), attention masks (`attention_mask`) and target labels (`labels`) are retained, as these are the inputs expected by the Hugging Face `Trainer`.

In [ ]:
tokenized_train = tokenized_train.remove_columns(
    ["id", "input", "standardized"]
)

tokenized_val = tokenized_val.remove_columns(
    ["id", "input", "standardized"]
)

#### ◆ Configure the data collator

The `DataCollatorForSeq2Seq` prepares batches during training by dynamically padding tokenized sequences to the length of the longest sequence within each batch. This ensures that batches have a consistent shape while avoiding unnecessary padding resulting in more efficient memory usage during training.

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

#### ◆ Training arguments (T4 GPU)

In [ ]:
training_args = TrainingArguments(
    output_dir="./byt5_results",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=5e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    fp16=True,
    report_to="none"
)

#### ◆ Create trainer

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

#### ◆ Start training

In [ ]:
start_time = time.time()
train_result = trainer.train()
end_time = time.time()
 
training_time = end_time - start_time

print(f"Training time: {training_time/60:.2f} minutes")

#### ◆ Extract and plot loss curves

In [ ]:
train_loss = []
train_epochs = []

val_loss = []
val_epochs = []

for log in trainer.state.log_history:

    if "loss" in log:
        train_loss.append(log["loss"])
        train_epochs.append(log["epoch"])

    if "eval_loss" in log:
        val_loss.append(log["eval_loss"])
        val_epochs.append(log["epoch"])

plt.figure(figsize=(8,5))

plt.plot(
    train_epochs,
    train_loss,
    label="Training loss"
)

plt.plot(
    val_epochs,
    val_loss,
    label="Validation loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("ByT5 training and validation loss")

plt.legend()
plt.grid()

plt.show()

#### ◆ Save loss history

In [ ]:
loss_history = {
    "train_epoch": train_epochs,
    "train_loss": train_loss,
    "val_epoch": val_epochs,
    "val_loss": val_loss
}

with open(
    "byt5_loss_history.json",
    "w"
) as f:

    json.dump(
        loss_history,
        f,
        indent=4
    )

#### ◆ Save ByT5 model after training

In [ ]:
trainer.save_model(
    "../../results/ByT5/final_model"
)

tokenizer.save_pretrained(
    "../../results/ByT5/final_model"
)

#### ◆ Tokenize the test dataset

In [ ]:
tokenized_test = test_dataset.map(
    preprocess_function,
    batched=True
)

tokenized_test = tokenized_test.remove_columns(
    ["id", "input", "standardized"]
)

#### ◆ Run inference and measure time

In [ ]:
start_time = time.time()
predictions = trainer.predict(tokenized_test)
end_time = time.time()

inference_time = end_time - start_time

print(f"Total inference time: {inference_time:.2f} seconds")
print(f"Average per sentence: {inference_time / len(test_data):.4f} seconds")

#### ◆ Decode predictions and references

In [ ]:
prediction_ids = predictions.predictions

# If predictions are returned as logits, convert them to token IDs
if prediction_ids.ndim == 3:
    prediction_ids = prediction_ids.argmax(axis=-1)

predicted_text = tokenizer.batch_decode(
    prediction_ids,
    skip_special_tokens=True
)

reference_text = tokenizer.batch_decode(
    predictions.label_ids,
    skip_special_tokens=True
)

#### ◆ Create results table

In [ ]:
results_byt5_df = pd.DataFrame({
    "Input": [item["input"] for item in test_data],
    "Reference": reference_text,
    "Prediction": predicted_text
})

results_byt5_df.head(10)

#### ◆ Save predictions

In [ ]:
results_byt5_df.to_csv(
    "byt5_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Predictions saved to byt5_predictions.csv")

Check for MAX LENGTH

Load dataset and ByT5 tokenizer

In [ ]:
import json
import numpy as np
import pandas as pd
from transformers import AutoTokenizer


# Load dataset
dataset_path = r"C:\Users\ae\OneDrive\thesis\data\data_versions\mc_joint_text_restoration.json"

with open(dataset_path, "r", encoding="utf-8") as f:
    data = json.load(f)


print("Total samples:", len(data))


# Load ByT5 tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "google/byt5-small"
)

2. Calculate ByT5 token lengths

In [ ]:
input_lengths = []
target_lengths = []


for item in data:

    # Input sentence
    input_tokens = tokenizer(
        item["input"],
        truncation=False
    )

    # Standardized target sentence
    target_tokens = tokenizer(
        item["standardized"],
        truncation=False
    )


    input_lengths.append(
        len(input_tokens["input_ids"])
    )

    target_lengths.append(
        len(target_tokens["input_ids"])
    )

sumarry stats

In [ ]:
import json
import pandas as pd


# Create summary statistics
input_stats = pd.Series(input_lengths).describe(
    percentiles=[0.90, 0.95, 0.99]
).to_dict()


target_stats = pd.Series(target_lengths).describe(
    percentiles=[0.90, 0.95, 0.99]
).to_dict()


4. Check how many examples exceed possible MAX_LENGTH values

In [ ]:
# Store MAX_LENGTH truncation analysis
truncation_analysis = {}

length_options = [128, 256, 512, 1024]


for limit in length_options:

    input_exceed = sum(
        length > limit
        for length in input_lengths
    )

    target_exceed = sum(
        length > limit
        for length in target_lengths
    )


    truncation_analysis[str(limit)] = {

        "input_exceeding": input_exceed,

        "input_exceeding_percentage":
            round(
                input_exceed / len(data) * 100,
                2
            ),

        "target_exceeding": target_exceed,

        "target_exceeding_percentage":
            round(
                target_exceed / len(data) * 100,
                2
            )
    }

Find examples which are too long

In [ ]:
long_examples = []


for idx, item in enumerate(data):

    length = len(
        tokenizer(
            item["input"],
            truncation=False
        )["input_ids"]
    )

    if length > 256:

        long_examples.append(
            {
                "id": item.get("id"),
                "token_length": length,
                "text": item["input"]
            }
        )


print(
    "Number of examples >256 tokens:",
    len(long_examples)
)


print(long_examples[:5])

Save all the summary in JSON file

In [ ]:
# Final JSON structure
byt5_length_analysis = {

    "title":
        "Summary statistics for ByT5 token length analysis to determine optimal MAX_LENGTH",

    "dataset":
        "mc_joint_text_restoration.json",

    "total_samples":
        len(data),

    "input_statistics":
        input_stats,

    "target_statistics":
        target_stats,

    "max_length_truncation_analysis":
        truncation_analysis
}


# -------------------------------
# Examples exceeding MAX_LENGTH=256
# -------------------------------

input_long_examples = []
target_long_examples = []


for item in data:

    # Check input length
    input_length = len(
        tokenizer(
            item["input"],
            truncation=False
        )["input_ids"]
    )

    if input_length > 256:

        input_long_examples.append(
            {
                "id": item.get("id"),
                "token_length": input_length,
                "text": item["input"]
            }
        )


    # Check target length
    target_length = len(
        tokenizer(
            item["standardized"],
            truncation=False
        )["input_ids"]
    )

    if target_length > 256:

        target_long_examples.append(
            {
                "id": item.get("id"),
                "token_length": target_length,
                "text": item["standardized"]
            }
        )


# Add long examples information to JSON
byt5_length_analysis["examples_exceeding_256_tokens"] = {

    "input_examples": {

        "description":
            "Input sentences where ByT5 token length exceeds MAX_LENGTH=256",

        "count":
            len(input_long_examples),

        "percentage":
            round(
                len(input_long_examples) / len(data) * 100,
                2
            ),

        "sample_examples":
            input_long_examples[:20]
    },


    "target_examples": {

        "description":
            "Target sentences where ByT5 token length exceeds MAX_LENGTH=256",

        "count":
            len(target_long_examples),

        "percentage":
            round(
                len(target_long_examples) / len(data) * 100,
                2
            ),

        "sample_examples":
            target_long_examples[:20]
    }
}


# -------------------------------
# Save JSON file
# -------------------------------

with open(
    "byt5_length_analysis.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        byt5_length_analysis,
        f,
        indent=4,
        ensure_ascii=False
    )

print("ByT5 length analysis saved successfully!")